<a href="https://colab.research.google.com/github/Gatoetkatja/patternRecognition/blob/main/PP_1_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**FUNGSI PREPROCESSING**
Fungsi Preprocessing yang digunakan berisi fungsi:
1. lowercase( ) untuk mentidakkapitalkan semua kata
2. rm_punctuation( ) untuk menghilangkan tanda baca
3. rm_numbers( ) untuk menghilangkan angka
4. tokenize( ) untuk memisah/tokenize kalimat menjadi token-token
5. rm_stopwords( ) untuk menghapus kata stopwords
6. rm_affix( ) untuk menghilangkan imbuhan



> Catatan 1:
Fungsi preprocessing tidak menghilangkan kata duplikat karena akan mengacaukan perhitungan Term Frequency.



> Catatan 2: Fungsi rm_affix sangatlah sederhana karena keterbatasan penggunaan library, jadi kemungkinan besar terjadi kesalahan ketika mencari kata dasar dari suatu kata





In [ ]:
import math
import string

def lowercase(text):
  text = text.lower()
  return text

def rm_punctuation(text: str) -> str:
  for char in text:
    if char in string.punctuation:
      text = text.replace(char, '')
  return text

def rm_numbers(text: str) -> str:
  for char in text:
    if char.isnumeric():
      text = text.replace(char, '')
  return text

def tokenize(text: str) -> list[str]: #Ini juga sekalian menghapus whitespace berlebih
  text = text.split()
  return text

def rm_duplicate(tokens: list[str]) -> list[str]:
  types = []
  for token in tokens:
    if token not in types:
       types.append(token)
  return types

stopwords = {'ada', 'adalah', 'adanya', 'akan', 'aku', 'atau', 'begitu', 'bahwa', 'bisa', 'dari',
             'dalam', 'dan', 'dengan', 'di', 'dia', 'ini', 'itu', 'jika', 'juga', 'kamu',
             'kami', 'ke', 'karena', 'kita', 'lagi', 'lah', 'maka', 'mereka', 'mau', 'namun',
             'oleh', 'pada', 'paling', 'pun', 'saja', 'saya', 'sangat', 'sebagai', 'sebuah', 'secara',
             'sedang', 'sehingga', 'serta', 'sudah', 'tapi', 'tentang', 'telah', 'tidak', 'untuk', 'yang'
             }

def rm_stopwords(types: list[str]) -> list[str]:
  return [t for t in types if t not in stopwords]

def rm_affix(word: str) -> str: #Masih sangat sederhana, masih sangat mungkin terjadi kesalahan saat penghilangan imbuhan
  prefixes = ['mem', 'menge', 'memper', 'meny', 'diper', 'memper', 'meng', 'me', 'di', 'ber', 'ter']
  suffixes = ['kan', 'an', 'nya', 'lah', 'in', 'mu', 'tah', 'pun', 'i']

  original = word

  for prefix in prefixes:
    if word.startswith(prefix) and len(word) - len(prefix) >= 3:
      word = word[len(prefix):]
      break

  for suffix in suffixes:
    if word.endswith(suffix) and len(word) - len(suffix) >= 3:
      word = word[:-len(suffix)]
      break

  return word if len(word) >= 3 else original

def preprocess(text) -> list[str]:
  text = lowercase(text)
  text = rm_punctuation(text)
  text = rm_numbers(text)

  tokens = tokenize(text)
  tokens = rm_stopwords(tokens)

  terms = [rm_affix(t) for t in tokens]

  return terms

#**FUNGSI PENGHITUNG TF, IDF, DAN TF-IDF**

In [ ]:
def compute_tf(terms: list[str]) -> dict[str, float]:
  tf = {}
  total = len(terms)
  if total == 0:
    return tf
  for term in terms:
    tf[term] = tf.get(term, 0) + 1
  for term in tf:
    tf[term] = tf[term] / total
  return tf

In [ ]:
def compute_idf(corpus_terms: list[list[str]]) -> dict[str, float]:

  total_doc = len(corpus_terms)
  df = {}

  for terms in corpus_terms:
    unique_terms = set(terms)
    for term in unique_terms:
      df[term] = df.get(term, 0) + 1

  idf = {}
  for term in df.keys():
    idf[term] = 1 + math.log(total_doc/df[term])

  return idf

In [ ]:
def compute_tfidf(tf: dict[str,float],
                  idf: dict[str, float]) -> dict[str,float]:
  tfidf = {}
  for term in tf.keys():
    tfidf[term] = tf[term] * idf.get(term, 0)
  return tfidf

#**DEMO PENGGUNAAN TF-IDF**

In [ ]:
corpus = [
    "Tulisan ini berisi latihan membaca buku.",
    "Bacaan ilmiah membantu latihan menulis.",
    "Tulisan ilmiah dimulai dari bacaan bermutu.",
    "Latihan menulis membantu pemahaman bacaan.",
]

# 1. Preprocessing
print("\nHASIL PREPROCESSING\n")
corpus_terms = []
for i, doc in enumerate(corpus):
    terms = preprocess(doc)
    corpus_terms.append(terms)
    print(f"  Dok {i+1}: {terms}")

# 2. TF
print("\nTERM FREQUENCY\n")
for i, terms in enumerate(corpus_terms):
    tf = compute_tf(terms)
    print(f"  Dok {i+1}:")
    for term, val in sorted(tf.items(), key=lambda x: -x[1]):
        print(f"    '{term}': {val:.4f}")

# 3. IDF
print("\nNVERSE DOCUMENT FREQUENCY\n")
idf = compute_idf(corpus_terms)
for term, val in sorted(idf.items(), key=lambda x: -x[1]):
    print(f"  '{term}': {val:.4f}")

# 4. TF-IDF
print("\nTF-IDF\n")
for i, terms in enumerate(corpus_terms):
    tf = compute_tf(terms)
    tfidf = compute_tfidf(tf, idf)
    top = sorted(tfidf.items(), key=lambda x: -x[1])
    top_str = ", ".join(f"'{t}'={v:.3f}" for t, v in top)
    print(f"  Dok {i+1}: {top_str}")



HASIL PREPROCESSING

  Dok 1: ['tulis', 'isi', 'latih', 'baca', 'buku']
  Dok 2: ['baca', 'ilmiah', 'bantu', 'latih', 'nulis']
  Dok 3: ['tulis', 'ilmiah', 'mula', 'baca', 'mutu']
  Dok 4: ['latih', 'nulis', 'bantu', 'pemaham', 'baca']

TERM FREQUENCY

  Dok 1:
    'tulis': 0.2000
    'isi': 0.2000
    'latih': 0.2000
    'baca': 0.2000
    'buku': 0.2000
  Dok 2:
    'baca': 0.2000
    'ilmiah': 0.2000
    'bantu': 0.2000
    'latih': 0.2000
    'nulis': 0.2000
  Dok 3:
    'tulis': 0.2000
    'ilmiah': 0.2000
    'mula': 0.2000
    'baca': 0.2000
    'mutu': 0.2000
  Dok 4:
    'latih': 0.2000
    'nulis': 0.2000
    'bantu': 0.2000
    'pemaham': 0.2000
    'baca': 0.2000

NVERSE DOCUMENT FREQUENCY

  'buku': 2.3863
  'isi': 2.3863
  'mula': 2.3863
  'mutu': 2.3863
  'pemaham': 2.3863
  'tulis': 1.6931
  'ilmiah': 1.6931
  'bantu': 1.6931
  'nulis': 1.6931
  'latih': 1.2877
  'baca': 1.0000

TF-IDF

  Dok 1: 'isi'=0.477, 'buku'=0.477, 'tulis'=0.339, 'latih'=0.258, 'baca'=0.200
  Do